### Gold - fact_pedido_item

Granularidade: 1 linha por item de pedido (order_id + item_seq), e não
por pedido. Essa granularidade permite agregação para o nível de pedido
(group by order_id), mantendo a capacidade de segmentação por produto e
categoria — requisito de negócio que seria perdido caso a fato fosse
salva apenas no nível de pedido.

O produto P8888, sem registro correspondente (notebook de produtos), é
incorporado via left join. O item permanece na fato, pois representa
receita real, sem os atributos de produto correspondentes, sinalizado
pela coluna produto_cadastrado = false.

In [ ]:
%run "../00_setup/00_setup_ambiente"

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

df_cab = spark.table(f"{schema_name}.silver_pedidos_cabecalho")
df_itens = spark.table(f"{schema_name}.silver_pedidos_itens")
df_dim_produto = spark.table(f"{schema_name}.dim_produto")
df_dim_cliente = spark.table(f"{schema_name}.dim_cliente")
df_dim_vendedor = spark.table(f"{schema_name}.dim_vendedor")
df_entregas = spark.table(f"{schema_name}.silver_entregas")

net_amount está disponível apenas no cabeçalho, não por item. Para
evitar duplicação de receita ao agregar por item, a receita é rateada
proporcionalmente ao peso de cada item dentro do pedido.

In [ ]:
w_pedido = Window.partitionBy("order_id")

df_itens_rateio = (
    df_itens
    .withColumn("total_item_pedido", F.sum("total_item").over(w_pedido))
    .withColumn("peso_item", F.when(F.col("total_item_pedido") > 0, F.col("total_item") / F.col("total_item_pedido")))
)

df_fato_base = (
    df_itens_rateio.alias("i")
    .join(df_cab.alias("c"), "order_id", "inner")
    .withColumn("receita_liquida_item", F.round(F.col("peso_item") * F.col("c.net_amount"), 2))
)

In [ ]:
df_fato_produto = (
    df_fato_base
    .join(df_dim_produto, df_fato_base.product_code == df_dim_produto.product_id, "left")
    .withColumn("produto_cadastrado", F.col("product_id").isNotNull())
)

print("Itens com produto não cadastrado:", df_fato_produto.filter(~F.col("produto_cadastrado")).count())

pedido_atrasado compara promised_date com a data de entrega real. Na
ausência de entrega registrada para o pedido, o valor permanece null —
distinto de "não atrasou".

In [ ]:
df_entregas_agg = (
    df_entregas.groupBy("order_id")
    .agg(
        F.max("delivered_at").alias("delivered_at_max"),
        F.first("delivery_status", ignorenulls=True).alias("delivery_status"),
        F.sum("cost").alias("custo_frete_total"),
    )
)

df_fato_entrega = (
    df_fato_produto
    .join(df_entregas_agg, "order_id", "left")
    .withColumn("pedido_cancelado", F.col("c.status_order") == "Cancelado")
    .withColumn(
        "pedido_atrasado",
        F.when(F.col("delivered_at_max").isNotNull(), F.to_date(F.col("delivered_at_max")) > F.col("c.promised_date"))
    )
    .withColumn("lead_time_prometido_dias", F.datediff(F.col("c.promised_date"), F.col("c.order_date")))
    .withColumn("lead_time_real_dias", F.datediff(F.to_date(F.col("delivered_at_max")), F.col("c.order_date")))
)

In [ ]:
df_fato_final_pre = (
    df_fato_entrega
    .join(df_dim_cliente.select("customer_id", "regiao", "segmento_cliente"), df_fato_entrega.customer_code == df_dim_cliente.customer_id, "left")
    .join(df_dim_vendedor.select("seller_id", "canal_venda", "tipo_canal"), "seller_id", "left")
)

df_fact_pedido_item = df_fato_final_pre.select(
    "order_id", "item_seq",
    F.col("customer_code").alias("customer_id"),
    "seller_id",
    F.col("product_code").alias("product_id"),
    F.col("c.order_date").alias("order_date"),
    F.col("c.promised_date").alias("promised_date"),
    F.col("delivered_at_max").alias("delivered_at"),
    F.col("c.status_order").alias("status_pedido"),
    "item_status", "quantity", "unit_price", "total_item", "receita_liquida_item",
    F.col("c.order_priority").alias("prioridade_pedido"),
    F.col("c.order_source").alias("canal_origem_pedido"),
    "regiao", "segmento_cliente", "canal_venda", "tipo_canal", "produto_cadastrado",
    "pedido_cancelado", "pedido_atrasado", "delivery_status", "custo_frete_total",
    "lead_time_prometido_dias", "lead_time_real_dias",
    F.col("c.flag_valor_inconsistente").alias("flag_valor_pedido_inconsistente"),
)

In [ ]:
# Validação: a soma da receita rateada por pedido deve reconciliar com net_amount do cabeçalho
df_check = (
    df_fact_pedido_item.groupBy("order_id").agg(F.sum("receita_liquida_item").alias("soma_itens"))
    .join(df_cab.select("order_id", "net_amount"), "order_id")
    .withColumn("diff", F.round(F.col("soma_itens") - F.col("net_amount"), 2))
    .filter(F.abs(F.col("diff")) > 0.05)
)
print("Pedidos com receita rateada divergente do cabeçalho (esperado: 0):", df_check.count())

In [ ]:
(
    df_fact_pedido_item.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("status_pedido")
    .saveAsTable(f"{schema_name}.fact_pedido_item")
)

print("fact_pedido_item:", df_fact_pedido_item.count(), "registros")